# Exploracao dos Dados BCB

Notebook para verificar integridade dos dados coletados do Banco Central do Brasil.

**Dados disponiveis:**
- **SGS Daily**: Selic, CDI, Dolar PTAX, Euro PTAX
- **SGS Monthly**: IBC-Br Bruto, IBC-Br Dessazonalizado, IGP-M
- **Expectations**: Expectativas do Relatorio Focus (IPCA, IGP-M, PIB, Cambio, Selic)

## 1. Setup e Carregamento

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Configurar matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10

# Caminho para dados
data_path = Path.cwd().parent / 'data'
processed_path = data_path / 'processed'

print(f"Data path: {data_path}")

In [ ]:
# Carregar todos os dados consolidados
df_sgs_daily = pd.read_parquet(processed_path / 'sgs_daily_consolidated.parquet')
df_sgs_monthly = pd.read_parquet(processed_path / 'sgs_monthly_consolidated.parquet')
df_expectations = pd.read_parquet(processed_path / 'expectations_consolidated.parquet')

print("Dados carregados com sucesso!")
print(f"  SGS Daily: {df_sgs_daily.shape[0]:,} registros x {df_sgs_daily.shape[1]} colunas")
print(f"  SGS Monthly: {df_sgs_monthly.shape[0]:,} registros x {df_sgs_monthly.shape[1]} colunas")
print(f"  Expectations: {df_expectations.shape[0]:,} registros x {df_expectations.shape[1]} colunas")

## 2. SGS Daily

In [ ]:
print("=" * 60)
print("SGS DAILY")
print("=" * 60)
print(f"\nShape: {df_sgs_daily.shape}")
print(f"Colunas: {list(df_sgs_daily.columns)}")
print(f"Periodo: {df_sgs_daily.index.min()} a {df_sgs_daily.index.max()}")
print(f"\nValores faltantes:")
print(df_sgs_daily.isna().sum())

In [ ]:
print("Primeiros registros:")
display(df_sgs_daily.head())
print("\nUltimos registros:")
display(df_sgs_daily.tail())

## 3. SGS Monthly

In [ ]:
print("=" * 60)
print("SGS MONTHLY")
print("=" * 60)
print(f"\nShape: {df_sgs_monthly.shape}")
print(f"Colunas: {list(df_sgs_monthly.columns)}")
print(f"Periodo: {df_sgs_monthly.index.min()} a {df_sgs_monthly.index.max()}")
print(f"\nValores faltantes:")
print(df_sgs_monthly.isna().sum())

In [ ]:
print("Primeiros registros:")
display(df_sgs_monthly.head())
print("\nUltimos registros:")
display(df_sgs_monthly.tail())

## 4. Expectations

In [ ]:
print("=" * 60)
print("EXPECTATIONS")
print("=" * 60)
print(f"\nShape: {df_expectations.shape}")
print(f"Colunas: {list(df_expectations.columns)}")
print(f"Periodo: {df_expectations.index.min()} a {df_expectations.index.max()}")
print(f"\nValores faltantes:")
print(df_expectations.isna().sum())

In [ ]:
# Contagem por fonte (_source)
print("Registros por indicador (_source):")
print(df_expectations['_source'].value_counts().sort_index())

In [ ]:
print("Primeiros registros:")
display(df_expectations.head())
print("\nUltimos registros:")
display(df_expectations.tail())

## 5. Cobertura Temporal

In [ ]:
def cobertura_temporal(df, nome):
    """Analisa a cobertura temporal de cada coluna."""
    print(f"\n{nome}")
    print("-" * 70)
    for col in df.columns:
        if col == '_source':
            continue
        dados = df[col].dropna()
        if len(dados) > 0:
            try:
                print(f"{col:30} | {dados.index.min().strftime('%Y-%m-%d')} a {dados.index.max().strftime('%Y-%m-%d')} | {len(dados):>8,} registros")
            except Exception:
                print(f"{col:30} | {len(dados):>8,} registros")
        else:
            print(f"{col:30} | SEM DADOS")

cobertura_temporal(df_sgs_daily, "SGS DAILY")
cobertura_temporal(df_sgs_monthly, "SGS MONTHLY")

In [ ]:
# Cobertura temporal por fonte de expectations
# Agora o indice e DatetimeIndex, entao podemos usar diretamente
print("\nEXPECTATIONS (por _source)")
print("-" * 70)

for source in sorted(df_expectations['_source'].unique()):
    subset = df_expectations[df_expectations['_source'] == source]
    min_date = subset.index.min().strftime('%Y-%m-%d')
    max_date = subset.index.max().strftime('%Y-%m-%d')
    print(f"{source:30} | {min_date} a {max_date} | {len(subset):>8,} registros")

## 6. Visualizacoes - SGS

In [ ]:
# Selic e CDI
fig, ax = plt.subplots(figsize=(14, 5))

# Identificar o início da série Selic para limitar o gráfico
start_date = df_sgs_daily['selic'].first_valid_index()
df_plot = df_sgs_daily.loc[start_date:]

if 'selic' in df_plot.columns:
    df_plot['selic'].dropna().plot(ax=ax, label='Meta Selic', linewidth=1)
if 'cdi_anualizado' in df_plot.columns:
    df_plot['cdi_anualizado'].dropna().plot(ax=ax, label='CDI Anualizado', linewidth=0.5, alpha=0.7)

ax.set_title('Taxa Selic e CDI')
ax.set_xlabel('Data')
ax.set_ylabel('Taxa (% a.a.)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Dolar e Euro PTAX
fig, ax = plt.subplots(figsize=(14, 5))

# Identificar o início da série Euro para limitar o gráfico
start_date = df_sgs_daily['euro_ptax'].first_valid_index()
df_plot = df_sgs_daily.loc[start_date:]

if 'dolar_ptax' in df_plot.columns:
    df_plot['dolar_ptax'].dropna().plot(ax=ax, label='Dolar PTAX', linewidth=1)
if 'euro_ptax' in df_plot.columns:
    df_plot['euro_ptax'].dropna().plot(ax=ax, label='Euro PTAX', linewidth=1)

ax.set_title('Taxas de Cambio PTAX')
ax.set_xlabel('Data')
ax.set_ylabel('R$')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IBC-Br
fig, ax = plt.subplots(figsize=(14, 5))

if 'ibc_br_bruto' in df_sgs_monthly.columns:
    df_sgs_monthly['ibc_br_bruto'].dropna().plot(ax=ax, label='IBC-Br Bruto', linewidth=1)
if 'ibc_br_dessaz' in df_sgs_monthly.columns:
    df_sgs_monthly['ibc_br_dessaz'].dropna().plot(ax=ax, label='IBC-Br Dessazonalizado', linewidth=1)

ax.set_title('IBC-Br - Indice de Atividade Economica')
ax.set_xlabel('Data')
ax.set_ylabel('Indice')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# IGP-M
fig, ax = plt.subplots(figsize=(14, 5))

# Filtrar dados a partir do ano 2000
df_plot = df_sgs_monthly.loc['2000':]

if 'igp_m' in df_plot.columns:
    df_plot['igp_m'].dropna().plot(ax=ax, color='green', linewidth=1)

ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_title('IGP-M - Variacao Mensal (%)')
ax.set_xlabel('Data')
ax.set_ylabel('Variacao (%)')
plt.tight_layout()
plt.show()

## 7. Visualizacoes - Expectations

In [ ]:
# Distribuicao de registros por fonte
fig, ax = plt.subplots(figsize=(12, 5))

counts = df_expectations['_source'].value_counts().sort_index()
counts.plot(kind='barh', ax=ax, color='steelblue')

ax.set_title('Registros por Indicador (Expectations)')
ax.set_xlabel('Quantidade de Registros')
ax.set_ylabel('Indicador')

# Adicionar valores nas barras
for i, v in enumerate(counts.values):
    ax.text(v + 1000, i, f'{v:,}', va='center')

plt.tight_layout()
plt.show()

In [ ]:
# Exemplo: IPCA Anual - Mediana ao longo do tempo
if 'ipca_anual' in df_expectations['_source'].values:
    ipca_anual = df_expectations[df_expectations['_source'] == 'ipca_anual'].copy()
    
    # Agrupar por data e calcular media da mediana
    if 'Mediana' in ipca_anual.columns:
        mediana_diaria = ipca_anual.groupby(ipca_anual.index)['Mediana'].mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        mediana_diaria.plot(ax=ax, linewidth=0.5)
        
        ax.set_title('IPCA Anual - Mediana das Expectativas')
        ax.set_xlabel('Data da Pesquisa')
        ax.set_ylabel('Expectativa (%)')
        plt.tight_layout()
        plt.show()
        
        print(f"Periodo: {ipca_anual.index.min()} a {ipca_anual.index.max()}")
        print(f"Registros: {len(ipca_anual):,}")

In [ ]:
# Exemplo: Selic - Mediana ao longo do tempo
if 'selic' in df_expectations['_source'].values:
    selic_exp = df_expectations[df_expectations['_source'] == 'selic'].copy()
    
    if 'Mediana' in selic_exp.columns:
        mediana_diaria = selic_exp.groupby(selic_exp.index)['Mediana'].mean()
        
        fig, ax = plt.subplots(figsize=(14, 5))
        mediana_diaria.plot(ax=ax, linewidth=0.5)
        
        ax.set_title('Selic - Mediana das Expectativas')
        ax.set_xlabel('Data da Pesquisa')
        ax.set_ylabel('Expectativa (%)')
        plt.tight_layout()
        plt.show()
        
        print(f"Periodo: {selic_exp.index.min()} a {selic_exp.index.max()}")
        print(f"Registros: {len(selic_exp):,}")

## 8. Resumo

Este notebook verificou a integridade dos dados coletados:

- **SGS Daily**: Series diarias de taxas de juros e cambio
- **SGS Monthly**: Series mensais de atividade economica e inflacao
- **Expectations**: Expectativas de mercado do Relatorio Focus

Para analises mais detalhadas (correlacoes, tendencias, previsoes), crie notebooks especificos.